In [ ]:
from finbourne_sdk_utils.jupyter_tools import toggle_code

"""Households

Demonstration of how to manage the holdings for an investor based on each Mandate & Household they are associated with.

Attributes
----------
instruments
portfolios
properties
set holdings
quotes
portfolio groups
aggregation
"""

toggle_code("Toggle Docstring")

## The Challenge

As a wealth manager you have a number of investors who are associated with multiple investment Mandates. Each Mandate may have one or more Accounts. These investors can be part of one or more Households, and a Household can contain one or more Investors. You would like to be able to see the holdings for an investor based on each Mandate & Household they are associated with. 

You also have a number of Branches and would like to see the assets managed by each Branch. Furthermore each branch has a number of Financial Advisors and you would like to be able to see the assets managed by each Financial Advisor.

## The Solution

1) Create your Instrument universe using a range of identifiers

2) Set up a scope for each branch to hold the accounts

3) Create a LUSID Portfolio for each account

4) Create a Property to hold the investor, mandate, and household details

5) Set the initial holdings of the portfolio

6) Load market data (prices)

7) Create a Portfolio Group for each household

8) Conduct a valuation against each household

9) Consider other ways of grouping & valuing client accounts

In [ ]:
# Import LUSID
import finbourne.sdk.services.lusid as lu
import finbourne.sdk.services.lusid.models as models
from finbourne.sdk.extensions import SyncApiClientFactory, RefreshingToken
from finbourne.sdk.exceptions import ApiException
import lusid_sample_data as import_data
from finbourne_sdk_utils.pandas_utils.lusid_pandas import lusid_response_to_data_frame

# Import Libraries
import pprint
from datetime import datetime, timedelta, time
import pytz
import printer as prettyprint
import pandas as pd
import numpy as np
import json
import uuid
import os

# Authenticate our user and create our API client
secrets_path = os.getenv("FBN_SECRETS_PATH")

api_factory = SyncApiClientFactory(
    access_token=RefreshingToken(),
    secrets_path=secrets_path,
    app_name="LusidJupyterNotebook",
)

print("LUSID Environment Initialised")
print(
    "LUSID SDK Version: ",
    api_factory.build(lu.ApplicationMetadataApi)
    .get_lusid_versions()
    .build_version,
)

![Scopes](img/paper-lusid.gif)

### Create your instrument universe using a range of identifiers

Before you can take on any holdings for your client accounts you need to ensure that your instrument universe has been populated. In this case you will import your instrument universe from a CSV file. Read more about instruments in LUSID in the [LUSID Knowledge Base: Instruments](https://support.lusid.com/what-is-an-instrument).

*Run the cell below to import your instrument universe*

In [ ]:
equity_instruments = pd.read_csv("data/households-instruments-equities.csv")
equity_instruments.head(n=20)

Now that you have the details for your instruments you can go ahead and create an instrument definition for each instrument. These can then be upserted into LUSID. Read about instrument definitions here [LUSID Knowledge Base: What is an Instrument?](https://support.lusid.com/what-is-an-instrument).

You use an upsert method to add instrument definitions to the instrument universe in LUSID. Read more about the behaviour of the upsert method here [LUSID Knowledge Base: Upsert](https://support.lusid.com/upsert-command).

For further usage of the upsert instruments API call refer to the [LUSID API Docs: Upserting Instruments](https://docs.lusid.com/#operation/UpsertInstruments).

*Run the cell below to upsert your instruments into LUSID*

In [ ]:
# Initialise your batch upsert request
batch_upsert_request = {}

# Using your instrument universe create your batch request
for index, instrument in equity_instruments.iterrows():

    # Specify the columns of your identifiers
    identifier_columns = ["QuotePermId", "Ticker", "ClientInternal"]

    # Create your identifiers
    identifiers = {}
    for identifier in identifier_columns:
        identifiers[identifier] = models.InstrumentIdValue(value=instrument[identifier])

    # Build your request and add it to the dictionary
    batch_upsert_request[instrument["InstrumentName"]] = models.InstrumentDefinition(
        name=instrument["InstrumentName"], identifiers=identifiers
    )

# Call LUSID to upsert your instrument definitions
instrument_response = api_factory.build(lu.InstrumentsApi).upsert_instruments(
    request_body=batch_upsert_request
)

# Pretty print the response
df = lusid_response_to_data_frame(list(instrument_response.values.values()))
df = df.loc[:, ~df.columns.str.startswith("href")]
df

![Scopes](img/households-instrumentmaster.gif)

### Set up a scope to hold the accounts

Your wealth managment company has a number of different branches. Each of these branches will be allocated a Scope in LUSID for its accounts. Read more about scopes in the [LUSID Knowledge Base: Scopes](https://support.lusid.com/what-is-a-scope-in-lusid-and-how-is-it-used).

*Run the cell below to create a name for the scope of two of your branches*

In [ ]:
# Get the ids for the scopes
scope_ids = [import_data.create_scope_id() for i in range(0, 3)]

# Set up a scope for each branch
branch_singapore_scope = "branch_{}".format(scope_ids[0])
branch_hongkong_scope = "branch_{}".format(scope_ids[1])
branch_scopes = [branch_singapore_scope, branch_hongkong_scope]

# Set up a scope for reporting
reporting_scope = "reporting_{}".format(scope_ids[2])

# Pretty print the responses
prettyprint.heading("Singapore Branch Scope", branch_singapore_scope)
prettyprint.heading("Hong Kong Branch Scope", branch_hongkong_scope)
prettyprint.heading("Reporting Scope", reporting_scope)

![Scopes](img/households-branch.gif)

### Create a portfolio for each account that exists with the branch

Now that you have decided on the name for your scope, you can create the portfolios to represent the client accounts inside this scope. You will import the account details from a CSV file.

*Run the cell below to import your client's account details*

In [ ]:
# Import the account details
accounts = pd.read_csv("data/households-accounts.csv")
accounts.head(n=10)

With the account details loaded you can now create your client portfolios.

Note that every portfolio can be referenced by a unique code. Read more about portfolios in the [LUSID Knowledge Base: Portfolios](https://support.lusid.com/what-is-2).

For further usage of the create portfolio API call refer to the [LUSID API Docs: Create Portfolio](https://docs.lusid.com/#operation/CreatePortfolio).

Note that when you create the portolios in the cell below you are creating it with a 'created' date of 1052 days ago. This number is rather arbitary, in practice it should be the date the portfolio came into existence regardless of the system you first created it in, read more about the importance of the created date on a portfolio in the [LUSID Knowledge Base: Importance of Portfolio Creation Date](https://support.lusid.com/importance-of-portfolio-creation-date).

*Run the cell below to create your portfolios*

In [ ]:
# A mapping between branch codes and scopes in LUSID
branch_mapping = {
    "Singapore_Singapore": branch_singapore_scope,
    "HongKong_SAR": branch_hongkong_scope,
}

responses = []

# Iterate over the client accounts
for index, account in accounts.iterrows():

    # Set the creation date of your client portfolio
    portfolio_creation_date = (datetime.now(pytz.UTC) - timedelta(days=1052)).isoformat()

    # Build your request to create your client portfolio
    request = models.CreateTransactionPortfolioRequest(
        display_name=account["account_name"],
        code=account["account_code"],
        base_currency=account["currency"],
        description=account["description"],
        created=portfolio_creation_date,
        corporate_action_source_id=None,
        accounting_method="AverageCost",
        sub_holding_keys=None,
        properties=None,
    )

    # Call LUSID to create your client portfolio
    responses.append(
        api_factory.build(lu.TransactionPortfoliosApi).create_portfolio(
            scope=branch_mapping[account["branch_id"]],
            create_transaction_portfolio_request=request,
        )
    )

# Pretty print the response
lusid_response_to_data_frame(responses)

![Scopes](img/households-portfolios.gif)

### Create a property to hold the investor and the household

To keep track of the investor associated with each portfolio as well as the household you can make use of LUSID's extensible properties. These allow you to define a bespoke schema for your portfolio objects. Read more about properties in the [LUSID Knowledge Base: Properties](https://support.lusid.com/what-is-a-property). 

For further usage of the create property definition API call refer to the [LUSID API Docs: Create Property Definition](https://docs.lusid.com/#operation/CreatePropertyDefinition).

*Run the cell below to create a property to hold household_id, the account owner ids & the financial advisor id*

In [ ]:
# The property codes to create
property_codes = [
    "household_id",
    "primary_acount_owner_id",
    "other_account_owner_id",
    "financial_advisor_id",
    "mandate_id",
    "mandate_description",
    "branch_id",
]

responses = []

# Iterate over the property codes
for property_code in property_codes:

    # Create your request to define a new property
    request = models.CreatePropertyDefinitionRequest(
        domain="Portfolio",
        scope=reporting_scope,
        code=property_code,
        value_required=False,
        display_name=property_code,
        data_type_id=models.ResourceId(scope="system", code="string"),
    )

    # Call LUSID to create your new property
    responses.append(
        api_factory.build(lu.PropertyDefinitionsApi).create_property_definition(
            create_property_definition_request=request
        )
    )

# Pretty print the response
df = lusid_response_to_data_frame(list(instrument_response.values.values()))
df = df.loc[:, ~df.columns.str.startswith("href")]
df

You can also create a property which can hold multiple values. These are known as "Collection" properties.

*Run the cell below to create a collection property to hold all account owners*

In [ ]:
# The collection property codes to create
property_codes_multi = {
    "account_owners": ["primary_acount_owner_id", "other_account_owner_id"]
}

responses = []

# Iterate over the property codes
for property_code in property_codes_multi.keys():

    # Create your request to define a new property
    request = models.CreatePropertyDefinitionRequest(
        domain="Portfolio",
        scope=reporting_scope,
        code=property_code,
        value_required=False,
        display_name=property_code,
        constraint_style="Collection",  # Note the constraint_style is set to "Collection"
        data_type_id=models.ResourceId(scope="system", code="string"),
    )

    # Call LUSID to create your new property
    responses.append(
        api_factory.build(lu.PropertyDefinitionsApi).create_property_definition(
            create_property_definition_request=request
        )
    )

# Pretty print the response
lusid_response_to_data_frame(responses)

With these properties created you can now populate them for each portfolio.

*Run the cell below to populate the values for these properties for each portfolio*

In [ ]:
# Iterate over each account
for index, account in accounts.iterrows():

    # Add the relevant "Collection" account properties to the portfolio
    response = api_factory.build(lu.PortfoliosApi).upsert_portfolio_properties(
        scope=branch_mapping[account["branch_id"]],
        code=account["account_code"],
        request_body={
            "Portfolio/{}/{}".format(
                reporting_scope, account_property
            ): models.ModelProperty(
                key="Portfolio/{}/{}".format(reporting_scope, account_property),
                value=models.PropertyValue(
                    label_value_set=models.LabelValueSet(
                        values=[
                            account[field]
                            for field in fields
                            if not pd.isnull(account[field])
                        ]
                    )
                ),
            )
            for account_property, fields in property_codes_multi.items()
        },
    )

    # Pretty print the response
    prettyprint.portfolio_properties_response(response)

    # Add the relevant Single Value account properties to the portfolio
    response = api_factory.build(lu.PortfoliosApi).upsert_portfolio_properties(
        scope=branch_mapping[account["branch_id"]],
        code=account["account_code"],
        request_body={
            "Portfolio/{}/{}".format(
                reporting_scope, account_property
            ): models.ModelProperty(
                key="Portfolio/{}/{}".format(reporting_scope, account_property),
                value=models.PropertyValue(label_value=account[account_property]),
            )
            for account_property in property_codes
            if not pd.isnull(account[account_property])
        },
    )

    # Pretty print the response
    prettyprint.portfolio_properties_response(response)
    print("\n")

![Scopes](img/households-properties.gif)

### Set the initial holdings of the portfolio

Now that you have your instrument universe populated and portfolios created you can load your current client holdings into their portfolios. In this case you will import their holdings from a CSV file. 

*Run the cell below to import your current client holdings*

In [ ]:
# Import and print the holdings
holdings = pd.read_csv("data/households-holdings.csv")
holdings.head(n=20)

Now that you have imported your client holdings you can add them to LUSID. You can do this by setting the holdings on a portfolio. Read more about how making an adjustment or setting the holdings on a portfolio affects it here [LUSID Knowledge Base: The effect of holding adjustments](https://support.lusid.com/how-do-holding-adjustments-affect-a-portfolio).

For further usage of the set holdings API call refer to the [LUSID API Docs: Set Holdings](https://docs.lusid.com/#operation/SetHoldings).

*Run the cell below to upsert your holdings into LUSID*

In [ ]:
# Make the holdings effective from now
holdings_effective_date = datetime.now(pytz.UTC).isoformat()

# Iterate over the portfolios in the holdings CSV
for portfolio in holdings["portfolio_code"].unique():

    # Initialise a list to hold your adjustments
    holding_adjustments = []

    # Iterate over the holdings in each portfolio
    for index, holding in holdings.loc[
        holdings["portfolio_code"] == portfolio
    ].iterrows():

        # Set your instrument identifiers based on whether or not instrument is cash
        if "Cash" in holding["instrument_name"]:
            identifier_key = "Instrument/default/Currency"
            identifer = holding["instrument_name"].split("_")[0]
        else:
            identifier_key = "Instrument/default/QuotePermId"
            identifer = holding["QuotePermId"]

        # Create your holding adjustment and append it to your list
        holding_adjustments.append(
            models.AdjustHoldingRequest(
                instrument_identifiers={identifier_key: identifer},
                tax_lots=[
                    models.TargetTaxLotRequest(
                        units=holding["quantity"],
                        cost=models.CurrencyAndAmount(
                            amount=holding["quantity"] * holding["price"],
                            currency=holding["currency"],
                        ),
                        portfolio_cost=holding["quantity"] * holding["price"],
                        price=holding["price"],
                    )
                ],
            )
        )

    # Call LUSID to set your initial holdings
    response = api_factory.build(lu.TransactionPortfoliosApi).set_holdings(
        scope=branch_mapping[holding["branch_id"]],
        code=portfolio,
        effective_at=holdings_effective_date,
        adjust_holding_request=holding_adjustments,
    )

    # Pretty print our response from LUSID
    prettyprint.set_holdings_response(
        response, branch_mapping[account["branch_id"]], portfolio
    )

### Load market data prices

With the Portfolio Groups created to group each household together, you can now aggregate across households.

To aggregate & value a portfolio in LUSID you need to upsert market data quotes against the underlying holdings or specify an analytics library to use. Read more about aggregating and valuing portfolios in the [LUSID Knowledge Base: Aggregations and Valuations](https://support.lusid.com/what-is-a-valuation).

In this case you will upsert market data quotes to the quote store to be used in an aggregation request. You will import these quotes from a CSV file.

*Run the cell below to import the market data prices*

In [ ]:
# Import the market data prices
prices = pd.read_csv("data/households-prices.csv")
prices.head(n=50)

Now that you have imported the market data you can add it to the quote store in LUSID. Read more about what a quote is in the [LUSID Knowledge Base: What is a Quote?](https://support.lusid.com/what-is-a-quote).

For further usage of the Upsert Quotes API call refer to the [LUSID API Docs: Upsert Quotes](https://docs.lusid.com/#operation/UpsertQuotes).

*Run the cell below to upsert the market data quotes into LUSID*

In [ ]:
# Initialise an empty list to hold the market data quotes
instrument_quotes = {}

# Iterate over each quote
for index, quote in prices.iterrows():

    # Get the LUSID Instrument ID for the quoted instrument
    luid = (
        api_factory.build(lu.SearchApi)
        .instruments_search(
            instrument_search_property=[
                models.InstrumentSearchProperty(
                    key="Instrument/default/QuotePermId", value=quote["QuotePermId"]
                )
            ],
            mastered_only=True,
        )[0]
        .mastered_instruments[0]
        .identifiers["LusidInstrumentId"]
        .value
    )

    # Create a quote for this instrument and append it to the list of quotes
    instrument_quotes[luid] = models.UpsertQuoteRequest(
        quote_id=models.QuoteId(
            quote_series_id=models.QuoteSeriesId(
                provider="DataScope",
                instrument_id=luid,
                instrument_id_type="LusidInstrumentId",
                quote_type="Price",
                field="Mid",
            ),
            effective_at=holdings_effective_date,
        ),
        metric_value=models.MetricValue(
            value=float(str(quote["price_current"]).replace(",", "")), unit=str(quote["currency"])
        ),
        lineage="InternalSystem",
    )

# Upsert the quotes into LUSID
response = api_factory.build(lu.QuotesApi).upsert_quotes(
    scope=reporting_scope, request_body=instrument_quotes
)

# Pretty print the response
lusid_response_to_data_frame(list(response.values.values()))

### Create a portfolio group for & conduct a valuation against each household

Now that you've created Portfolios for the client accounts, you can group them together into households using Portfolio Groups.

Read more about portfolio groups here [LUSID Knowledge Base: How do you Group and Aggregate Portfolios?](https://support.lusid.com/how-do-you-group-and-aggregate-portfolios)

*Run the cell below to create a function which allows you to create the portfolio groups and add the relevant portfolios*

In [ ]:
def create_portfolio_groups(account_data, grouping_key, group_scope, portfolio_scopes):
    """
    param: account_data (DataFrame) - The Pandas DataFrame with the account data
    param: grouping_key (str) - The key to group the accounts by
    param: group_scope (str) - The scope to create the portfolio group in
    param: portfolio_scopes (list[str]) - The list of scopes that contain the
    accounts to group

    returns: N/A
    """

    group_creation_date = (datetime.now(pytz.UTC) - timedelta(days=5000)).isoformat()
    portfolio_creation_date = (datetime.now(pytz.UTC) - timedelta(days=1052)).isoformat()

    # Raise an error saying that the key does not exist in the data
    if grouping_key not in account_data.columns:

        raise (
            """
            The grouping key does not exist in the account data! Please
            check your spelling and that the key exists in the data and 
            try again
            """
        )

    # Get all the unique groups from the grouping key
    for group in account_data[grouping_key].unique():

        # Build a create group request for this group
        group_request = models.CreatePortfolioGroupRequest(
            code=group + "-Group",
            display_name="Contains all accounts for the {} of {}".format(
                grouping_key, group
            ),
            created=group_creation_date,
        )

        try:

            # Call LUSID to create the portfolio group
            response = api_factory.build(
                lu.PortfolioGroupsApi
            ).create_portfolio_group(
                scope=group_scope, create_portfolio_group_request=group_request
            )

            # Pretty print the response
            prettyprint.portfolio_group_response(response, "created")

        except ApiException as e:
            if json.loads(e.body)["code"] == 128:
                pass
            else:
                print(json.loads(e.body)["title"])

    # Initialise a list to hold all account portfolios
    portfolios = []

    # Iterate over all the scopes
    for portfolio_scope in portfolio_scopes:

        # Call LUSID to list all the portfolios across all the portfolio scopes
        response = api_factory.build(lu.PortfoliosApi).list_portfolios_for_scope(
            scope=portfolio_scope
        )

        # Loop over each portfolio
        for portfolio in response.values:

            # For this portfolio get its properties which you defined earlier
            properties = api_factory.build(
                lu.PortfoliosApi
            ).get_portfolio_properties(scope=portfolio.id.scope, code=portfolio.id.code)

            # Make the list of properties easy to work with by converting them to key-value pairs
            portfolio_properties = properties.properties

            try:

                # Using the group property determine which portfolio group to add this portfolio too
                response = api_factory.build(
                    lu.PortfolioGroupsApi
                ).add_portfolio_to_group(
                    scope=group_scope,
                    code=portfolio_properties[
                        "Portfolio/{}/{}".format(group_scope, grouping_key)
                    ].value.label_value
                    + "-Group",
                    resource_id=models.ResourceId(
                        scope=portfolio.id.scope, code=portfolio.id.code
                    ),
                    effective_at=portfolio_creation_date,
                )

                # Pretty print the response
                prettyprint.get_portfolio_group_response(response)

            except ApiException as e:
                if json.loads(e.body)["code"] == 171:
                    pass
                else:
                    print(json.loads(e.body)["title"])

Now that you've defined your function you can use it to create portfolio groups for each household.

*Run the cell below to group the portfolios by household*

In [ ]:
create_portfolio_groups(
    account_data=accounts,
    grouping_key="household_id",
    group_scope=reporting_scope,
    portfolio_scopes=branch_scopes,
)

Using these portfolio groups you can perform an aggregation across each household. The logic for an aggregation is controled by a LUSID recipe. Read more about recipes in the [LUSID Knowledge Base: What is a Recipe and How Are They Used?](https://support.lusid.com/what-is-a-recipe-and-how-are-they-used).

For further usage of the Get Aggregation by Portfolio API call refer to the [LUSID API Docs: Get Aggregation by Portfolio](https://docs.lusid.com/#operation/GetAggregationByPortfolio).

*Run the cell below to create a function to aggregate and value each household's investments*

In [ ]:
recipe_code = "market_value"
recipe_scope = "households"

# Create a recipe to perform a valuation
configuration_recipe_api = api_factory.build(lu.ConfigurationRecipeApi)

configuration_recipe = models.ConfigurationRecipe(
    scope=recipe_scope,
    code=recipe_code,
    market=models.MarketContext(
        market_rules=[
            models.MarketDataKeyRule(
                key="Quote.LusidInstrumentId.*",
                supplier="DataScope",
                data_scope=reporting_scope,
                quote_type="Price",
                field="Mid",
            )
        ],
        suppliers=models.MarketContextSuppliers(
            commodity="DataScope",
            credit="DataScope",
            equity="DataScope",
            fx="DataScope",
            rates="DataScope",
        ),
        options=models.MarketOptions(
            default_supplier="DataScope",
            default_instrument_code_type="LusidInstrumentId",
            default_scope=reporting_scope,
        ),
    ),
)

upsert_configuration_recipe_response = (
    configuration_recipe_api.upsert_configuration_recipe(
        upsert_recipe_request=models.UpsertRecipeRequest(
            configuration_recipe=configuration_recipe
        )
    )
)

In [ ]:
recipe = models.ResourceId(scope=recipe_scope, code=recipe_code)


def aggregate_portfolio_group(grouping_key, group_scope, recipe=recipe):
    """
    param: grouping_key (str) - The key to group the accounts by
    param: group_scope (str) - The scope to the portfolio group is in

    returns: aggregation_results (list[DataFrame]) - The list of resulting
    aggregation dataframes
    """

    # Initialse a list to hold the aggregation results
    aggregation_results_df = []

    # Iterate over each household
    for group in accounts[grouping_key].unique():

        # Create the valuation request
        valuation_request = models.ValuationRequest(
            recipe_id=recipe,
            metrics=[
                models.AggregateSpec(
                    key="Instrument/default/LusidInstrumentId", op="Value"
                ),
                models.AggregateSpec(key="Holding/default/Cost", op="Sum"),
                models.AggregateSpec(key="Valuation/PvInReportCcy", op="Sum"),
                models.AggregateSpec(key="Holding/default/Units", op="Sum"),
                models.AggregateSpec(key="Instrument/default/Name", op="Value"),
            ],
            group_by=["Instrument/default/Name"],
            portfolio_entity_ids=[
                models.PortfolioEntityId(
                    scope=group_scope,
                    code=group + "-Group",
                    portfolio_entity_type="GroupPortfolio",
                )
            ],
            valuation_schedule=models.ValuationSchedule(
                effective_at=holdings_effective_date
            ),
        )

        # Perform a valuation
        valuation = api_factory.build(lu.AggregationApi).get_valuation(
            valuation_request=valuation_request
        )

        # Pretty print the response
        aggregation_results_df.append(
            prettyprint.aggregation_response_generic_df(
                response=valuation, index_key="Instrument/default/Name", name=group
            )
        )

    return aggregation_results_df

Now that you've defined your function you can use it to aggregate across each household.

*Run the cell below to aggregate across each household*

In [ ]:
aggregation_results_df = aggregate_portfolio_group(
    grouping_key="household_id", group_scope=reporting_scope
)

In [ ]:
prettyprint.heading("Household", aggregation_results_df[0].columns.name)
aggregation_results_df[0]

In [ ]:
prettyprint.heading("Household", aggregation_results_df[1].columns.name)
aggregation_results_df[1]

![Scopes](img/households-portfoliogroupshouseholds.gif)

### Group & aggregate by financial advisor

Perhaps you might also like to group your client accounts by financial advisor. Using your functions you defined earlier you can do this as well.

*Run the cell below to create groups for each advisor and perform aggregations*

In [ ]:
# Create the portfolio groups for each financial advisor
create_portfolio_groups(
    account_data=accounts,
    grouping_key="financial_advisor_id",
    group_scope=reporting_scope,
    portfolio_scopes=branch_scopes,
)

# Aggregate across the portfolio groups
aggregation_results_df = aggregate_portfolio_group(
    grouping_key="financial_advisor_id", group_scope=reporting_scope
)

In [ ]:
prettyprint.heading("Financial Advisor", aggregation_results_df[0].columns.name)
aggregation_results_df[0]

In [ ]:
prettyprint.heading("Financial Advisor", aggregation_results_df[1].columns.name)
aggregation_results_df[1]

In [ ]:
prettyprint.heading("Financial Advisor", aggregation_results_df[2].columns.name)
aggregation_results_df[2]

![Scopes](img/households-portfoliogroupsadvisor.gif)

### Group & aggregate by investment mandate

You can even group and aggregate by investment mandate. 

*Run the cell below to group and aggregate by investment mandate*

In [ ]:
# Create the portfolio groups for each mandate
create_portfolio_groups(
    account_data=accounts,
    grouping_key="mandate_id",
    group_scope=reporting_scope,
    portfolio_scopes=branch_scopes,
)

# Aggregate across the portfolio groups
aggregation_results_df = aggregate_portfolio_group(
    grouping_key="mandate_id", group_scope=reporting_scope
)

In [ ]:
prettyprint.heading("Mandate", aggregation_results_df[0].columns.name)
aggregation_results_df[0]

In [ ]:
prettyprint.heading("Mandate", aggregation_results_df[1].columns.name)
aggregation_results_df[1]

In [ ]:
prettyprint.heading("Mandate", aggregation_results_df[2].columns.name)
aggregation_results_df[2]

In [ ]:
prettyprint.heading("Mandate", aggregation_results_df[3].columns.name)
aggregation_results_df[3]

![Scopes](img/households-portfoliogroupsmandate.gif)

### Group & aggregate by branch

You can also create a portfolio group to see the investment across all the client accounts across each branch.

*Run the cell below to create a portfolio group for each branch*

In [ ]:
# Create the portfolio groups for each mandate
create_portfolio_groups(
    account_data=accounts,
    grouping_key="branch_id",
    group_scope=reporting_scope,
    portfolio_scopes=branch_scopes,
)

# Aggregate across the portfolio groups
aggregation_results_df = aggregate_portfolio_group(
    grouping_key="branch_id", group_scope=reporting_scope
)

In [ ]:
prettyprint.heading("Branch", aggregation_results_df[0].columns.name)
aggregation_results_df[0]

In [ ]:
aggregation_results_df

In [ ]:
prettyprint.heading("Branch", aggregation_results_df[1].columns.name)
aggregation_results_df[1]

![Scopes](img/households-portfoliogroupsscopes.gif)

Finally, you can also create a portfolio group to see the investment across all the client accounts across all branches.

In this case instead of using the functions you defined earlier you will add the group for each branch as sub-groups to the all accounts group.

*Run the cell below to create a portfolio group for all accounts*

In [ ]:
# Build a create group request for this group
group_request = models.CreatePortfolioGroupRequest(
    code="allaccounts-Group",
    display_name="Contains all accounts for all branches",
    created=(datetime.now(pytz.UTC) - timedelta(days=5000)).isoformat(),
)

# Call LUSID to create the portfolio group
response = api_factory.build(lu.PortfolioGroupsApi).create_portfolio_group(
    scope=reporting_scope, create_portfolio_group_request=group_request
)

# Pretty print the response
prettyprint.portfolio_group_response(response, "created")

In [ ]:
# Iterate over all the scopes
for branch in accounts["branch_id"].unique():

    # Call LUSID to add the sub-group to the all accounts group
    response = api_factory.build(lu.PortfolioGroupsApi).add_sub_group_to_group(
        scope=reporting_scope,
        code="allaccounts-Group",
        resource_id=models.ResourceId(
            scope=reporting_scope, code="{}-Group".format(branch)
        ),
        effective_at=(datetime.now(pytz.UTC) - timedelta(days=5000)).isoformat(),
    )

    # Pretty print the response
    prettyprint.get_portfolio_group_response(response)

In [ ]:
# Create valuation request
valuation_request = models.ValuationRequest(
    recipe_id=recipe,
    metrics=[
        models.AggregateSpec(key="Instrument/default/LusidInstrumentId", op="Value"),
        models.AggregateSpec(key="Holding/default/Cost", op="Sum"),
        models.AggregateSpec(key="Valuation/PvInReportCcy", op="Sum"),
        models.AggregateSpec(key="Holding/default/Units", op="Sum"),
        models.AggregateSpec(key="Instrument/default/Name", op="Value"),
    ],
    group_by=["Instrument/default/Name"],
    portfolio_entity_ids=[
        models.PortfolioEntityId(
            scope=reporting_scope,
            code="allaccounts-Group",
            portfolio_entity_type="GroupPortfolio",
        )
    ],
    valuation_schedule=models.ValuationSchedule(
        effective_at=holdings_effective_date
    ),
)

# Perform a valuation
valuation_response = api_factory.build(lu.AggregationApi).get_valuation(
    valuation_request=valuation_request
)

aggregation_results_df = prettyprint.aggregation_response_generic_df(
    response=valuation_response,
    index_key="Instrument/default/Name",
    name="All Accounts",
)

In [ ]:
prettyprint.heading("All Accounts", "")
aggregation_results_df

![Scopes](img/households-groupsall.gif)